# DeepSeekMoE: many small experts, a few that are always on

*DeepSeek-V3 technical report, section 2.1.2. Notebook 2 of 8.*

**TL;DR.** A mixture-of-experts (MoE) layer replaces one big feed-forward network by many small ones and sends each token to only a few of them, so the model can hold many more parameters than it spends compute on. DeepSeekMoE makes three design choices: experts are **small and numerous** (256 of them, 8 used per token), **one expert is shared** and sees every token, and routing is **limited to a few nodes** so that moving tokens between GPUs stays cheap. This notebook builds the layer from scratch, checks it against a deliberately naive version, measures what each choice does on small problems, and is explicit about what a toy cannot tell us.

| step | what we do |
|---|---|
| 1 | where DeepSeek-V3's 671B parameters sit, and why only 37B are used per token |
| 2 | the router: sigmoid scores, top-k, normalised gates |
| 3 | why many small experts beat few big ones (counting combinations) |
| 4 | shared experts, tested on a problem built to need them |
| 5 | node-limited routing |
| 6 | no token dropping |
| 7 | the finished layer, verified |
| 8 | dense versus DeepSeekMoE in a small language model |

The balance of load between experts, which the paper handles with a bias term instead of a loss, is the subject of notebook 3.

In [1]:
import sys, math, time
from pathlib import Path
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd()))             # the notebook lives next to dsv3kit.py
import dsv3kit as K                             # tested building blocks; this notebook re-shows the code it uses

torch.manual_seed(0)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .25, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})
C = dict(blue="#1f78b4", orange="#d95f02", green="#1b9e77", purple="#7570b3", red="#c0392b", grey="#777777")

PASS, FAIL = [], []
def check(name, ok, detail=""):
    (PASS if ok else FAIL).append(name)
    print(("PASS" if ok else "FAIL"), "-", name, detail)

print("torch", torch.__version__)

torch 2.14.1


## 1. Where the parameters are

A transformer block has an attention part and a feed-forward part (FFN). In large models the FFN holds most of the weights. An MoE layer keeps the FFN idea but splits it into $N_r$ separate **experts**. For every token a **router** picks $K_r$ of them, and only those run. Parameters grow with $N_r$; compute grows with $K_r$.

DeepSeek-V3's numbers (paper, section 4.2): hidden size $d=7168$, the first 3 of 61 layers keep an ordinary FFN, and the other 58 are MoE layers with 1 shared expert and 256 routed experts of hidden size 2048, of which 8 are chosen per token. Each expert is a SwiGLU FFN with three matrices (gate, up, down). Let's add it up.

In [2]:
d, d_ff_exp, n_layers, n_dense = 7168, 2048, 61, 3
n_moe = n_layers - n_dense
per_expert = 3 * d * d_ff_exp                              # gate (d x ff) + up (d x ff) + down (ff x d)

routed_all = n_moe * 256 * per_expert                      # every routed expert in the model
shared_all = n_moe * 1 * per_expert
active_experts = n_moe * (8 + 1) * per_expert              # 8 routed + 1 shared run for each token

with torch.device("meta"):
    attn = K.MLA(d=7168, n_h=128, d_h=128, d_c=512, d_cq=1536, d_r=64)
attn_all = n_layers * sum(p.numel() for p in attn.parameters())

print(f"one expert                       : {per_expert/1e6:7.1f} M parameters")
print(f"all routed experts (58 layers)   : {routed_all/1e9:7.1f} B")
print(f"all shared experts               : {shared_all/1e9:7.1f} B")
print(f"attention (61 layers, MLA)       : {attn_all/1e9:7.1f} B")
print(f"expert weights used per token    : {active_experts/1e9:7.1f} B   (8 routed + 1 shared, 58 layers)")
bottom_up = routed_all + shared_all + attn_all
print(f"\nbottom-up total of the above      : {bottom_up/1e9:.1f} B   (paper: 671 B total, 37 B activated per token)")
print(f"experts are {100 * (routed_all + shared_all) / 671e9:.0f}% of the 671 B; the rest (embeddings, 3 dense layers, MTP module, norms) is about {(671e9 - bottom_up)/1e9:.0f} B")
print(f"expert + attention weights touched per token: {(active_experts + attn_all)/1e9:.1f} B  vs 37 B activated in the paper")
check("one expert is 44.0M parameters", abs(per_expert / 1e6 - 44.04) < 0.01)
check("routed experts are about 650B of the 671B", 640e9 < routed_all < 660e9)
check("bottom-up total does not exceed the paper's 671B", bottom_up < 671e9)
check("activated experts + attention is below the paper's 37B (embeddings and dense layers make up the rest)", active_experts + attn_all < 37e9)

one expert                       :    44.0 M parameters
all routed experts (58 layers)   :   653.9 B
all shared experts               :     2.6 B
attention (61 layers, MLA)       :    11.4 B
expert weights used per token    :    23.0 B   (8 routed + 1 shared, 58 layers)

bottom-up total of the above      : 667.9 B   (paper: 671 B total, 37 B activated per token)
experts are 98% of the 671 B; the rest (embeddings, 3 dense layers, MTP module, norms) is about 3 B
expert + attention weights touched per token: 34.4 B  vs 37 B activated in the paper
PASS - one expert is 44.0M parameters 
PASS - routed experts are about 650B of the 671B 
PASS - bottom-up total does not exceed the paper's 671B 
PASS - activated experts + attention is below the paper's 37B (embeddings and dense layers make up the rest) 


Only the expert and attention weights can be computed from numbers the paper states; embeddings and the three dense layers need sizes it does not list in the text, so the last lines bound the 37B rather than reproduce it. The point stands: the 671B is about 98% expert weights, and each token touches about 1/28 of them.

## 2. The router

Notation follows the paper (eqs. 12 to 15). The input of the layer for token $t$ is $\mathbf u_t$. Each routed expert $i$ owns a **centroid vector** $\mathbf e_i$. The router scores how well the token matches each centroid:

$$ s_{i,t} = \sigma(\mathbf u_t^\top \mathbf e_i) \qquad\text{(affinity, sigmoid)} $$

Then it keeps the $K_r$ highest scores, and turns the kept scores into gate weights by normalising them to sum to one:

$$ g'_{i,t} = \begin{cases} s_{i,t} & s_{i,t}\in \mathrm{Top}K_r \\ 0 & \text{otherwise}\end{cases}, \qquad g_{i,t} = \frac{g'_{i,t}}{\sum_j g'_{j,t}} . $$

The layer output is the shared experts plus the gated routed experts (the residual connection $\mathbf u_t$ is added by the block):

$$ \mathbf h'_t = \sum_{i=1}^{N_s}\mathrm{FFN}^{(s)}_i(\mathbf u_t) + \sum_{i=1}^{N_r} g_{i,t}\,\mathrm{FFN}^{(r)}_i(\mathbf u_t). $$

![Paper Figure 2, upper right box: the router scores the input, Top-K selects routed experts, shared experts always run.](../../assets/deepseek_v3/fig2_architecture.png)

*Figure 2 of the paper. The upper right box is DeepSeekMoE: green experts are shared and always used, blue experts are routed, and the router picks the Top-$K_r$ of them.*

**Sigmoid instead of softmax.** DeepSeek-V2 used a softmax over all experts; V3 switches to an independent sigmoid per expert and normalises only over the chosen ones. With a softmax the scores compete: raising one expert's score lowers the others'. With sigmoids each expert's affinity stands alone, and the normalisation happens after the choice. A small example:

In [3]:
logits = torch.tensor([3.0, 2.5, 2.4, -1.0, -1.0, -1.0, -1.0, -1.0])      # token likes experts 0, 1, 2
top = 2
soft = logits.softmax(0)
sig = torch.sigmoid(logits)

def gates(scores, k):
    idx = scores.topk(k).indices
    g = scores[idx]
    return idx.tolist(), (g / g.sum()).tolist()

print("softmax scores :", [round(float(x), 3) for x in soft])
print("sigmoid scores :", [round(float(x), 3) for x in sig])
print("top-2 gates, softmax-then-normalise:", gates(soft, top))
print("top-2 gates, sigmoid-then-normalise:", gates(sig, top))
# adding a 4th strong expert changes the softmax scores of the OTHER experts but not their sigmoid scores
logits2 = logits.clone(); logits2[3] = 3.0
print("score of expert 0 when expert 3 becomes strong:  softmax %.3f -> %.3f   sigmoid %.3f -> %.3f" %
      (soft[0], logits2.softmax(0)[0], sig[0], torch.sigmoid(logits2)[0]))
check("sigmoid affinities are independent of the other experts", float(torch.sigmoid(logits2)[0]) == float(sig[0]))
check("softmax affinities are not", abs(float(logits2.softmax(0)[0]) - float(soft[0])) > 0.05)
check("normalised gates sum to one", abs(sum(gates(sig, top)[1]) - 1) < 1e-6)

softmax scores : [0.445, 0.27, 0.244, 0.008, 0.008, 0.008, 0.008, 0.008]
sigmoid scores : [0.953, 0.924, 0.917, 0.269, 0.269, 0.269, 0.269, 0.269]
top-2 gates, softmax-then-normalise: ([0, 1], [0.622459352016449, 0.3775406777858734])
top-2 gates, sigmoid-then-normalise: ([0, 1], [0.5075750350952148, 0.49242499470710754])
score of expert 0 when expert 3 becomes strong:  softmax 0.445 -> 0.310   sigmoid 0.953 -> 0.953
PASS - sigmoid affinities are independent of the other experts 
PASS - softmax affinities are not 
PASS - normalised gates sum to one 


Now the layer itself, built twice. First the slowest, most literal version: loop over tokens, loop over the chosen experts, exactly as in the equations. Then the efficient version that the rest of the notebooks use, which groups tokens by expert. If both give the same numbers we know the efficient one is right.

An expert is an ordinary SwiGLU feed-forward network, `down(silu(gate(x)) * up(x))`:

In [4]:
class Expert(nn.Module):
    """One FFN expert (SwiGLU): down( silu(gate(x)) * up(x) )."""

    def __init__(self, d: int, d_ff: int):
        super().__init__()
        self.gate = nn.Linear(d, d_ff, bias=False)
        self.up = nn.Linear(d, d_ff, bias=False)
        self.down = nn.Linear(d_ff, d, bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))

In [5]:
def moe_literal(x, shared, routed, centroids, top_k):
    """Paper eqs. (12)-(15), one token at a time. No batching tricks."""
    out = torch.zeros_like(x)
    for t in range(x.shape[0]):
        u = x[t]
        h = sum(sh(u) for sh in shared)                              # shared experts: always
        s = torch.sigmoid(centroids @ u)                             # (15) affinity to every routed expert
        chosen = s.topk(top_k).indices                               # (14) the K_r best
        g = s[chosen] / s[chosen].sum()                              # (13) normalise over the chosen
        for gi, e in zip(g, chosen):
            h = h + gi * routed[int(e)](u)                           # (12) gated sum
        out[t] = h
    return out

torch.manual_seed(0)
d_toy, n_routed, top_k = 16, 8, 2
x_toy = torch.randn(6, d_toy)
shared_toy = nn.ModuleList([Expert(d_toy, 12)])
routed_toy = nn.ModuleList([Expert(d_toy, 12) for _ in range(n_routed)])
cent_toy = torch.randn(n_routed, d_toy) * d_toy ** -0.5

with torch.no_grad():
    y_lit = moe_literal(x_toy, shared_toy, routed_toy, cent_toy, top_k)
    print("affinities of token 0 :", [round(float(v), 2) for v in torch.sigmoid(cent_toy @ x_toy[0])])
    print("chosen experts (all 6 tokens):", [torch.sigmoid(cent_toy @ x_toy[t]).topk(top_k).indices.tolist() for t in range(6)])
print("literal output shape:", tuple(y_lit.shape))

affinities of token 0 : [0.33, 0.32, 0.69, 0.36, 0.22, 0.47, 0.63, 0.41]
chosen experts (all 6 tokens): [[2, 6], [4, 2], [5, 0], [0, 7], [6, 0], [5, 0]]
literal output shape: (6, 16)


The efficient layer does two things differently. It computes all scores at once, and instead of looping over tokens it loops over **experts**: for each expert it gathers the tokens that chose it, runs them as one batch, and adds the result back weighted by the gate. Each expert therefore sees only its own tokens, which is where the compute saving comes from. This also contains the bias term of the load-balancing method (notebook 3); it is zero by default and only affects *which* experts are chosen, never the gate values.

In [6]:
class DeepSeekMoE(nn.Module):
    """Shared experts + fine-grained routed experts (paper eqs. 12-16).

    n_groups / topk_groups implement node-limited routing: experts are split into n_groups "nodes";
    every token may only use experts from its topk_groups best nodes.
    """

    def __init__(self, d, d_ff, n_shared, n_routed, top_k, n_groups=1, topk_groups=1):
        super().__init__()
        self.top_k, self.n_routed, self.n_groups, self.topk_groups = top_k, n_routed, n_groups, topk_groups
        self.shared = nn.ModuleList(Expert(d, d_ff) for _ in range(n_shared))
        self.routed = nn.ModuleList(Expert(d, d_ff) for _ in range(n_routed))
        self.centroids = nn.Parameter(torch.randn(n_routed, d) * d ** -0.5)          # e_i in eq. (15)
        self.register_buffer("bias", torch.zeros(n_routed))                           # b_i in eq. (16): routing only
        self.last = {}

    def route(self, u):
        aff = torch.sigmoid(u @ self.centroids.t())                                   # s_{i,t}, eq. (15)
        sel = aff.detach() + self.bias                                                # what top-k looks at, eq. (16)
        if self.n_groups > 1:                                                         # node-limited routing
            N = u.shape[0]
            per = self.n_routed // self.n_groups
            k_in = self.top_k // self.topk_groups
            group_score = sel.view(N, self.n_groups, per).topk(k_in, dim=-1).values.sum(-1)
            keep = group_score.topk(self.topk_groups, dim=-1).indices
            ok = torch.zeros(N, self.n_groups, dtype=torch.bool, device=u.device).scatter(1, keep, True)
            sel = sel.masked_fill(~ok.repeat_interleave(per, dim=1), float("-inf"))
        idx = sel.topk(self.top_k, dim=-1).indices                                    # chosen experts
        g = aff.gather(1, idx)                                                        # gate = ORIGINAL affinity (no bias)
        g = g / g.sum(-1, keepdim=True)                                               # eq. (13)
        return aff, idx, g

    def forward(self, x):
        B, T, d = x.shape
        u = x.reshape(-1, d)
        aff, idx, g = self.route(u)
        out = torch.zeros_like(u)
        for sh in self.shared:                                                        # always-on experts
            out = out + sh(u)
        for e in range(self.n_routed):                                                # only the chosen tokens
            tok, slot = (idx == e).nonzero(as_tuple=True)
            if tok.numel():
                out.index_add_(0, tok, g[tok, slot].unsqueeze(-1) * self.routed[e](u[tok]))
        self.last = dict(aff=aff, idx=idx, load=torch.bincount(idx.reshape(-1), minlength=self.n_routed))
        return out.view(B, T, d)

In [7]:
moe = DeepSeekMoE(d_toy, 12, n_shared=1, n_routed=n_routed, top_k=top_k)
moe.shared, moe.routed, moe.centroids.data = shared_toy, routed_toy, cent_toy          # same weights as the literal version
with torch.no_grad():
    y_eff = moe(x_toy.unsqueeze(0))[0]
print("max |literal - efficient| =", float((y_lit - y_eff).abs().max()))
check("efficient layer equals the literal equations", torch.allclose(y_lit, y_eff, atol=1e-5))
check("every token was sent to exactly top_k experts", int(moe.last["load"].sum()) == 6 * top_k)

max |literal - efficient| = 1.1920928955078125e-07
PASS - efficient layer equals the literal equations 
PASS - every token was sent to exactly top_k experts 


## 3. Why many small experts: counting combinations

Take a fixed budget of expert parameters and compute per token. You can spend it on a few big experts, or on many experts each a fraction of the size, activating proportionally more of them. Compute per token is the same. What changes is how many *different combinations* the router can pick from. Here is the paper's reasoning in numbers:

In [8]:
from math import comb
print(f"{'setup':45s} {'experts':>8s} {'top-k':>6s} {'size':>6s} {'combinations':>16s}")
for name, n, k, size in [("coarse: 16 experts, 2 per token", 16, 2, "1x"),
                         ("split each in 4: 64 experts, 8 per token", 64, 8, "1/4"),
                         ("split each in 16: 256 experts, 32 per token", 256, 32, "1/16"),
                         ("DeepSeek-V3: 256 experts, 8 per token", 256, 8, "1/16*")]:
    print(f"{name:45s} {n:8d} {k:6d} {size:>6s} {comb(n, k):16,d}")
print("* V3 uses small experts but activates fewer of them than a full split would; the router has many more options than with 16 big experts.")
check("64-choose-8 is far larger than 16-choose-2", comb(64, 8) > 1e6 * comb(16, 2))
check("V3's 256-choose-8 is about 4e14", 4e14 < comb(256, 8) < 5e14)

setup                                          experts  top-k   size     combinations
coarse: 16 experts, 2 per token                     16      2     1x              120
split each in 4: 64 experts, 8 per token            64      8    1/4    4,426,165,368
split each in 16: 256 experts, 32 per token        256     32   1/16 58,244,594,029,230,756,747,090,036,884,923,424,763,000
DeepSeek-V3: 256 experts, 8 per token              256      8  1/16* 409,663,695,276,000
* V3 uses small experts but activates fewer of them than a full split would; the router has many more options than with 16 big experts.
PASS - 64-choose-8 is far larger than 16-choose-2 
PASS - V3's 256-choose-8 is about 4e14 


With 16 big experts and 2 per token the router can build 120 different "mixes". Cutting every expert into four and activating eight per token costs the same compute, but allows about 4.4 billion mixes. A token can recombine small specialists more precisely. This is the argument from the DeepSeekMoE paper that V3 inherits; combinations are not quality, so the argument says what is *possible*, not what a trained model does. The small experiment in step 8 is where we look for an effect.

## 4. Shared experts

Some knowledge is needed by every token: function words, punctuation, basic syntax. In an ordinary MoE the routed experts would each have to learn their own copy. DeepSeekMoE sets aside $N_s$ **shared experts** that every token passes through, so the routed experts are free to specialise. V3 uses one.

We test this on a problem built to need it, so the effect, if there is one, is visible. Each input has a domain label $z\in\{0,\dots,7\}$ (a one-hot vector appended to the input) and the target is

$$ \mathbf y = A\,\mathbf x \;+\; B_z\,\mathbf x ,$$

where $A$ is one **common** matrix used by every token and $B_z$ is a **domain-specific** one. Two layers with the same compute per token:

- **routed only:** 8 experts, each token uses 2;
- **shared + routed:** 1 shared expert, 8 routed experts, each token uses 1 routed. Same two expert-evaluations per token; one more expert of parameters.

(Both use the bias-based balancing of notebook 3 so that the router does not collapse onto a few experts.)

In [9]:
def make_problem(seed, n_dom=8, d_x=16):
    g = torch.Generator().manual_seed(seed)
    A = torch.randn(d_x, d_x, generator=g) / d_x ** .5
    B = 0.7 * torch.randn(n_dom, d_x, d_x, generator=g) / d_x ** .5
    def batch(n, gen):
        x = torch.randn(n, d_x, generator=gen)
        z = torch.randint(0, n_dom, (n,), generator=gen)
        y = x @ A.T + torch.einsum("nij,nj->ni", B[z], x)
        return torch.cat([x, F.one_hot(z, n_dom).float()], -1), y
    return batch

class Toy(nn.Module):
    def __init__(self, n_shared, top_k, d_in=24, d_out=16, d_ff=32, n_routed=8):
        super().__init__()
        self.moe = DeepSeekMoE(d_in, d_ff, n_shared, n_routed, top_k)
        self.out = nn.Linear(d_in, d_out)
    def forward(self, x):
        return self.out(x + self.moe(x.unsqueeze(1)).squeeze(1))

def fit(n_shared, top_k, seed, steps=1000, gamma=0.005):
    torch.manual_seed(seed)
    batch = make_problem(seed=100)                      # the SAME problem for every run; seeds change init and data order
    gen = torch.Generator().manual_seed(seed)
    net = Toy(n_shared, top_k)
    opt = torch.optim.Adam(net.parameters(), lr=3e-3)
    for _ in range(steps):
        x, y = batch(256, gen)
        loss = F.mse_loss(net(x), y)
        opt.zero_grad(); loss.backward(); opt.step()
        K.update_bias(net.moe, gamma)
    xv, yv = batch(4096, torch.Generator().manual_seed(999))
    with torch.no_grad():
        return float(F.mse_loss(net(xv), yv)), sum(p.numel() for p in net.parameters())

t0 = time.time()
res4 = {}
for name, (ns, k) in {"routed only (2 of 8)": (0, 2), "1 shared + 1 of 8": (1, 1)}.items():
    runs = [fit(ns, k, s) for s in range(3)]
    res4[name] = np.array([r[0] for r in runs]); n_par = runs[0][1]
    print(f"{name:22s} val MSE {res4[name].mean():.4f} +- {res4[name].std():.4f}   (3 seeds)   parameters {n_par}")
print("time %.0f s;  target variance %.3f" % (time.time() - t0, float(make_problem(100)(4096, torch.Generator().manual_seed(999))[1].var())))

routed only (2 of 8)   val MSE 0.0021 +- 0.0002   (3 seeds)   parameters 19024


1 shared + 1 of 8      val MSE 0.0259 +- 0.0013   (3 seeds)   parameters 21328
time 31 s;  target variance 1.468


In [10]:
a, b = res4["routed only (2 of 8)"], res4["1 shared + 1 of 8"]
print(f"shared-expert variant is {100 * (1 - b.mean() / a.mean()):+.0f}% lower in MSE than routed-only (seed spreads: {a.std():.4f}, {b.std():.4f})")
check("both variants learn the task (MSE far below the target variance)", a.mean() < 0.3 * 1.0 and b.mean() < 0.3 * 1.0, "(target variance is about 1.5)")

shared-expert variant is -1138% lower in MSE than routed-only (seed spreads: 0.0002, 0.0013)
PASS - both variants learn the task (MSE far below the target variance) (target variance is about 1.5)


Read this result for what it is. The two models have the same number of expert evaluations per token, but the shared variant also has one more expert of parameters, so a gain mixes "shared" with "a bit larger". The test is also built to favour sharing, with a component that truly is common to all inputs. It tells us the mechanism works when the structure is there; it does not tell us how much real language gains, and the paper does not isolate it at V3 scale either (it inherits the choice from DeepSeekMoE and DeepSeek-V2).

## 5. Node-limited routing

At scale, experts live on different GPUs and a token's hidden vector has to be **sent** to every GPU that holds one of its chosen experts, and the results **sent back**. That all-to-all traffic is expensive, especially between machines. V3's cluster has 8 GPUs per machine (a "node") with a fast link inside a node and a slower one between nodes.

DeepSeek's rule: each token may use experts on at most $M = 4$ nodes. How a token picks its nodes (paper, "Node-Limited Routing"): score each node by the **sum of the $K_r/M$ highest affinities** among its experts, keep the best $M$ nodes, and then do the normal top-$K_r$ among the experts of those nodes. With $K_r=8$ and $M=4$ that is the sum of each node's top-2 affinities.

We use the layer's `n_groups` / `topk_groups` options (one "group" is one node) with V3's shapes: 256 experts in 8 nodes of 32, 8 experts per token, at most 4 nodes. The affinities below are random, so this measures the rule, not a trained router.

In [11]:
torch.manual_seed(0)
big = DeepSeekMoE(d=32, d_ff=4, n_shared=0, n_routed=256, top_k=8, n_groups=8, topk_groups=4)
free = DeepSeekMoE(d=32, d_ff=4, n_shared=0, n_routed=256, top_k=8, n_groups=1, topk_groups=1)
free.centroids.data = big.centroids.data.clone()
u = torch.randn(5000, 32)

with torch.no_grad():
    aff_b, idx_b, g_b = big.route(u)
    aff_f, idx_f, g_f = free.route(u)
nodes_b = torch.tensor([len(set((r // 32).tolist())) for r in idx_b])
nodes_f = torch.tensor([len(set((r // 32).tolist())) for r in idx_f])
print(f"nodes a token talks to:  unrestricted mean {nodes_f.float().mean():.2f} (max {int(nodes_f.max())})   limited to M=4: mean {nodes_b.float().mean():.2f} (max {int(nodes_b.max())})")

mass = lambda aff, idx: aff.gather(1, idx).sum(1)
print(f"affinity mass kept by the restricted choice: {float((mass(aff_b, idx_b) / mass(aff_f, idx_f)).mean()):.3f} of the unrestricted top-8")
print(f"tokens whose chosen set differs from the unrestricted one: {100 * float((idx_b.sort(1).values != idx_f.sort(1).values).any(1).float().mean()):.0f}%")

# check the node-scoring rule against a manual computation for token 0
sel = aff_b[0].view(8, 32)
manual_scores = sel.topk(2, dim=-1).values.sum(-1)                      # K_r / M = 2 highest per node
manual_nodes = set(manual_scores.topk(4).indices.tolist())
check("a limited token never uses more than 4 nodes", int(nodes_b.max()) <= 4)
check("an unrestricted token usually spreads over more than 4 nodes", float(nodes_f.float().mean()) > 5)
check("chosen nodes match the paper's 'sum of top K_r/M affinities' rule", set((idx_b[0] // 32).tolist()) <= manual_nodes)

nodes a token talks to:  unrestricted mean 5.31 (max 8)   limited to M=4: mean 3.97 (max 4)
affinity mass kept by the restricted choice: 0.994 of the unrestricted top-8
tokens whose chosen set differs from the unrestricted one: 86%
PASS - a limited token never uses more than 4 nodes 
PASS - an unrestricted token usually spreads over more than 4 nodes 
PASS - chosen nodes match the paper's 'sum of top K_r/M affinities' rule 


Random scores spread an unrestricted token over about 5 of the 8 nodes on average (up to 8); the rule caps it at 4 and costs only a few percent of the affinity mass, which is what a router would be pulled to give up to save communication. A trained router's affinities are not random, so the real cost of the restriction is something the paper handles by training with the constraint; we only verify the rule here. The paper also states that, combined with its communication kernels, each token reaches 3.2 experts per node on average and the layer can run with computation and communication fully overlapped. We cannot test that on one machine.

## 6. No token dropping

Classic MoE layers give every expert a fixed **capacity**, a maximum number of tokens per batch, to keep GPUs evenly busy. Tokens beyond an expert's capacity are **dropped**: they skip the layer. That wastes data and makes the model behave differently in training and inference.

V3 does not drop tokens, because its load balancing (notebook 3) keeps experts evenly used. To see what dropping would mean when balance is poor, take the layer from step 2 with a router that favours a few experts, and count how many token slots exceed a capacity of `capacity_factor * average load`:

In [12]:
torch.manual_seed(0)
moe_d = DeepSeekMoE(d=32, d_ff=4, n_shared=0, n_routed=64, top_k=4)
u = torch.randn(8192, 32)
def dropped(bias_scale, cf):
    moe_d.bias.copy_(bias_scale * torch.randn(64))                       # a skewed router: some experts are favoured
    with torch.no_grad():
        _, idx, _ = moe_d.route(u)
    load = torch.bincount(idx.reshape(-1), minlength=64).float()
    cap = cf * load.sum() / 64
    return float((load - cap).clamp_min(0).sum() / load.sum()), float(load.max() / load.mean())

def balance_with_bias(gamma=0.01, iters=400):
    """Run the notebook-3 control loop on fresh batches until the load is even (preview of the next notebook)."""
    moe_d.bias.zero_()
    for _ in range(iters):
        with torch.no_grad():
            _, idx, _ = moe_d.route(torch.randn(8192, 32))
        moe_d.last = dict(load=torch.bincount(idx.reshape(-1), minlength=64))
        K.update_bias(moe_d, gamma)

def dropped_now(cf):
    with torch.no_grad():
        _, idx, _ = moe_d.route(u)
    load = torch.bincount(idx.reshape(-1), minlength=64).float()
    cap = cf * load.sum() / 64
    return float((load - cap).clamp_min(0).sum() / load.sum()), float(load.max() / load.mean())

print(f"{'router':>28s} {'max load / mean':>16s} {'dropped @ cf=1.0':>18s} {'dropped @ cf=1.25':>19s}")
rows6 = {}
for name, setup in [("untrained, no balancing", lambda: moe_d.bias.zero_()),
                    ("skewed (bias noise 0.3)", lambda: moe_d.bias.copy_(0.3 * torch.randn(64))),
                    ("after the bias control loop", lambda: balance_with_bias())]:
    setup()
    d1, imb = dropped_now(1.0); d2, _ = dropped_now(1.25)
    rows6[name] = (d1, d2, imb)
    print(f"{name:>28s} {imb:16.2f} {100*d1:17.1f}% {100*d2:18.1f}%")
check("a skewed router drops a large share of tokens at capacity factor 1.25", rows6["skewed (bias noise 0.3)"][1] > 0.3)
check("after balancing by the bias loop, almost nothing is dropped at 1.25", rows6["after the bias control loop"][1] < 0.03, "(%.1f%%)" % (100 * rows6["after the bias control loop"][1]))

                      router  max load / mean   dropped @ cf=1.0   dropped @ cf=1.25
     untrained, no balancing             2.06              15.0%                6.9%
     skewed (bias noise 0.3)            13.85              74.1%               70.6%


 after the bias control loop             1.23               3.2%                0.0%
PASS - a skewed router drops a large share of tokens at capacity factor 1.25 
PASS - after balancing by the bias loop, almost nothing is dropped at 1.25 (0.0%)


## 7. The finished layer, checked

Before using the layer in a language model, three more checks on the class from step 2: gradient flow (only the experts that were chosen should receive a gradient for a given token), the parameter and compute accounting, and that gates stay normalised under node-limited routing.

In [13]:
torch.manual_seed(0)
layer = DeepSeekMoE(d=24, d_ff=16, n_shared=1, n_routed=8, top_k=2)
x1 = torch.randn(1, 1, 24, requires_grad=True)                          # ONE token
layer(x1).sum().backward()
chosen = set(layer.last["idx"][0].tolist())
has_grad = {e for e in range(8) if layer.routed[e].gate.weight.grad is not None and layer.routed[e].gate.weight.grad.abs().sum() > 0}
shared_grad = layer.shared[0].gate.weight.grad.abs().sum() > 0
cent_rows = {e for e in range(8) if layer.centroids.grad[e].abs().sum() > 0}
print("experts chosen for the token       :", sorted(chosen))
print("routed experts that got a gradient :", sorted(has_grad))
print("centroid rows that got a gradient  :", sorted(cent_rows))
check("only chosen routed experts receive gradient", has_grad == chosen)
check("the shared expert always receives gradient", bool(shared_grad))
check("only chosen centroids receive gradient (the gate is normalised over chosen experts)", cent_rows == chosen)

n_exp = sum(p.numel() for p in layer.parameters() if p.dim() == 2) - layer.centroids.numel()
per = 3 * 24 * 16
print(f"layer holds {9} experts x {per} = {9*per} weights; one token uses (1 shared + 2 routed) x {per} = {3*per} ({100*3/9:.0f}%)")
_, idx2, g2 = DeepSeekMoE(d=24, d_ff=8, n_shared=0, n_routed=16, top_k=4, n_groups=4, topk_groups=2).route(torch.randn(50, 24))
check("gates sum to one under node-limited routing too", torch.allclose(g2.sum(-1), torch.ones(50), atol=1e-5))

experts chosen for the token       : [1, 4]
routed experts that got a gradient : [1, 4]
centroid rows that got a gradient  : [1, 4]
PASS - only chosen routed experts receive gradient 
PASS - the shared expert always receives gradient 
PASS - only chosen centroids receive gradient (the gate is normalised over chosen experts) 
layer holds 9 experts x 1152 = 10368 weights; one token uses (1 shared + 2 routed) x 1152 = 3456 (33%)
PASS - gates sum to one under node-limited routing too 


## 8. Dense versus DeepSeekMoE in a small language model

Last, the layers inside a real (small) language model. The two models below use the same attention and the same compute per token in the feed-forward part (192 hidden units evaluated per token), but the MoE one holds three times as many feed-forward parameters:

- **dense**: an ordinary SwiGLU FFN with 192 hidden units;
- **DeepSeekMoE**: 1 shared expert and 8 routed experts of 64 hidden units, 2 routed per token ($64 + 2\times 64 = 192$ active). As in the paper, the first layer stays dense.

Character-level, 96-wide, 3 layers, two seeds, 300 steps of 48 sequences of 64 characters. The MoE run uses the bias balancing of notebook 3 with $\gamma = 0.01$.

In [14]:
data = K.CharData()
base = dict(vocab=data.vocab, d=96, n_layers=3, n_heads=4, attn="mha")
cfgs = {"dense FFN (192)": K.LMConfig(**base, ffn="dense", d_ff=192),
        "DeepSeekMoE (1 shared + 2/8 routed, 64 each)": K.LMConfig(**base, ffn="moe", d_ff=192, n_shared=1, n_routed=8, top_k=2, moe_ff=64)}
res8 = {}
t0 = time.time()
for name, cfg in cfgs.items():
    vals = []
    for seed in range(2):
        model, _ = K.train_lm(cfg, data, steps=300, B=48, T=64, lr=4e-3, seed=seed, gamma=0.01 if cfg.ffn == "moe" else 0.0)
        vals.append(K.evaluate(model, data, T=64, n_batches=10)["main"])
    n_par = sum(p.numel() for p in model.parameters())
    ffn_par = sum(p.numel() for b in model.blocks for p in b.ffn.parameters())
    res8[name] = (np.array(vals), n_par, ffn_par)
    print(f"{name:46s} val loss {np.mean(vals):.3f} +- {np.std(vals):.3f}   total params {n_par/1e6:.2f}M   FFN params {ffn_par/1e6:.2f}M")
print("time %.0f s" % (time.time() - t0))
load = model.moes()[0].last["load"].float()
print("expert load in the last batch (layer 1):", [int(v) for v in load], " max/mean = %.2f" % float(load.max() / load.mean()))

dense FFN (192)                                val loss 2.449 +- 0.009   total params 0.29M   FFN params 0.17M


DeepSeekMoE (1 shared + 2/8 routed, 64 each)   val loss 2.502 +- 0.038   total params 0.51M   FFN params 0.39M
time 85 s
expert load in the last batch (layer 1): [286, 243, 304, 215, 271, 224, 249, 256]  max/mean = 1.19


In [15]:
(vd, _, _), (vm, _, _) = list(res8.values())
print(f"MoE minus dense: {vm.mean() - vd.mean():+.3f} nats (seed spreads {vd.std():.3f} and {vm.std():.3f})")
check("both models learn", vd.mean() < math.log(data.vocab) - 1.5 and vm.mean() < math.log(data.vocab) - 1.5)
check("no expert is starved or flooded in the last batch (max load < 3x mean)", float(load.max() / load.mean()) < 3)

MoE minus dense: +0.053 nats (seed spreads 0.009 and 0.038)
PASS - both models learn 
PASS - no expert is starved or flooded in the last batch (max load < 3x mean) 


## What to take away

1. **MoE decouples parameters from compute.** V3 has 671B parameters, about 98% of them expert weights, and each token uses about 1/28 of the expert weights.
2. **DeepSeekMoE = fine-grained routed experts + always-on shared experts.** Small experts give the router many more combinations; shared experts take the common work so routed experts can specialise.
3. **Sigmoid affinities, normalised after the top-k choice** (V3's change from V2's softmax): an expert's score does not depend on how good the other experts are.
4. **Node-limited routing** caps how many machines a token touches (here 4 of 8), at a small cost in affinity mass. It exists for communication, not for quality.
5. **The layer is verified** against a literal loop of the paper's equations, for gradient flow, and for gate normalisation.

## What this notebook does not show
- **The quality benefit of the design at scale.** Our toy problems tell us the mechanisms work (the shared-expert test is built to need it) and that a tiny MoE language model trains; at 300 steps and two seeds they cannot rank DeepSeekMoE against a dense layer in general. Compare the gaps to the printed seed spreads.
- **That DeepSeekMoE helps.** In our language-model run the MoE model ended about 0.05 nats *worse* than the dense one, about the size of its seed spread; its three times larger FFN was not turned into a better loss in 300 steps.
- **Anything about speed on GPUs.** Our layer loops over experts in Python. The real training system fuses and overlaps these operations across 2048 GPUs.
- **Trained-router behaviour under node limits.** The node-limited test used random affinities.

*Next:* notebook 3 explains how V3 keeps the experts evenly loaded without an auxiliary loss.

In [16]:
print("passed:", len(PASS), "| failed:", len(FAIL))
assert not FAIL, FAIL

passed: 23 | failed: 0
